# Module `graph_generator.py`

Ce notebook genere une premiere instance CESIPATH a partir des hypotheses du livrable : graphe connexe, demandes uniformes, restrictions statiques, puis completion par plus courts chemins.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.graph_generator import GraphGenerator
from cesipath.models import GraphGenerationConfig

In [ ]:
config = GraphGenerationConfig(
    node_count=7,
    seed=42,
    edge_density=None,
    auto_density_profile=True,
    forbidden_rate=0.1,
    surcharge_rate=0.25,
    vehicle_capacity=40,
    dynamic_forbid_probability=0.03,
    dynamic_restore_probability=0.2,
    dynamic_max_disabled_ratio=0.2,
)
generator = GraphGenerator(config)
instance = generator.generate()

pprint(instance.summary())
print("\nDemandes :", instance.demands)
print("\nCoordonnees :")
pprint(instance.coordinates)
print("\nDensite cible base :", config.resolved_edge_density)
print("\nDensite base :", instance.base_density)
print("Densite residuelle :", instance.residual_density)
print("Degre moyen residuel :", instance.residual_average_degree)
print("\nMatrice du graphe de base :")
for row in instance.base_costs:
    print(row)
print("\nMatrice du graphe residuel (contraintes appliquees) :")
for row in instance.residual_costs:
    print(row)
print("\nMatrice des couts complets :")
for row in instance.completed_costs:
    print(row)

In [ ]:
snapshot = generator.initialize_dynamic_snapshot(instance)
print("\nEtat dynamique initial - matrice residuelle :")
for row in snapshot.residual_costs:
    print(row)
print("\nEtat dynamique initial - graphe complet :")
for row in snapshot.completed_costs:
    print(row)
print("\nDisponibilite initiale des vraies aretes :")
pprint(snapshot.edge_availability)

next_snapshot = generator.advance_dynamic_snapshot(instance, snapshot)
print("\nApres un deplacement - matrice residuelle dynamique :")
for row in next_snapshot.residual_costs:
    print(row)
print("\nApres un deplacement - graphe complet recalcule :")
for row in next_snapshot.completed_costs:
    print(row)
print("\nDisponibilite apres un deplacement :")
pprint(next_snapshot.edge_availability)

Le notebook permet de verifier rapidement trois points clefs :

- le graphe de base montre uniquement la topologie et les couts initiaux ;
- le graphe residuel affiche clairement l'effet des contraintes statiques, avec suppression des aretes interdites et surcout sur les autres ;
- le generateur rejette automatiquement les instances trop creuses, trop denses ou avec un degre moyen residuel insuffisant ;
- l'etat dynamique fait ensuite evoluer les vraies aretes autorisees, y compris leur disponibilite temporaire ;
- la matrice finale reste complete grace a Dijkstra applique apres chaque mise a jour dynamique, ce qui conserve une fermeture metrique pour la resolution.